In [1]:
''' # =========================================================
# 🔥 1. IMPORTS
# =========================================================
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder


# =========================================================
# 📦 2. LOAD DATASET
# =========================================================
df = pd.read_csv("dataset_fixed.csv")


# =========================================================
# 🔤 3. ENCODING (FOR ML MODEL)
# =========================================================
le_attack = LabelEncoder()
le_priv = LabelEncoder()
le_product = LabelEncoder()

df["attack_vector_enc"] = le_attack.fit_transform(df["attack_vector"])
df["privileges_enc"] = le_priv.fit_transform(df["privileges"])
df["product_enc"] = le_product.fit_transform(df["product"])


# =========================================================
# 🤖 4. TRAIN ML MODEL (RISK PREDICTION)
# =========================================================
X = df[["cvss", "attack_vector_enc", "privileges_enc", "product_enc"]]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

print("✅ ML Model Trained")


# =========================================================
# 🧠 5. NLP INPUT PARSER
# =========================================================
def parse_input(user_input):
    parts = user_input.lower().split()
    
    product = parts[0]
    version = parts[1] if len(parts) > 1 else None
    
    return product, version


# =========================================================
# 🔢 6. VERSION HANDLING (SMART 🔥)
# =========================================================

# Normalize version (12.0 → 12.0.0)
def normalize(v, length=3):
    try:
        parts = [int(x) for x in v.split(".")]
        while len(parts) < length:
            parts.append(0)
        return tuple(parts)
    except:
        return (0,)

# Generate fallback versions (12.0.1 → [12.0.1, 12.0, 12])
def get_generic_versions(v):
    parts = v.split(".")
    
    versions = []
    versions.append(v)  # exact
    
    if len(parts) > 1:
        versions.append(".".join(parts[:-1]))  # 12.0
    
    versions.append(parts[0])  # 12
    
    return list(set(versions))


# =========================================================
# 🔍 7. CVE FINDER (CORE ENGINE)
# =========================================================
def find_cves(product_input, version_input):
    results = df[df["product"].str.lower() == product_input.lower()]
    
    matched = []
    
    user_versions = get_generic_versions(version_input)
    
    for _, row in results.iterrows():
        try:
            row_version = str(row["version"])
            
            for v in user_versions:
                user_v = normalize(v)
                row_v = normalize(row_version)
                
                # Match conditions
                if row["operator"] == "=" and user_v == row_v:
                    matched.append(row)
                
                elif row["operator"] == "<=" and user_v <= row_v:
                    matched.append(row)
                
                elif row["operator"] == "<" and user_v < row_v:
                    matched.append(row)
                
                elif row["operator"] == ">=" and user_v >= row_v:
                    matched.append(row)
                
                elif row["operator"] == ">" and user_v > row_v:
                    matched.append(row)
        
        except:
            continue
    
    return pd.DataFrame(matched).drop_duplicates()


# =========================================================
# 🧠 8. ML RANKING (RECOMMENDATION ENGINE)
# =========================================================
def rank_cves(cve_df):
    if cve_df.empty:
        return cve_df
    
    temp = cve_df.copy()
    
    # Encode categorical features
    temp["attack_vector_enc"] = le_attack.transform(temp["attack_vector"])
    temp["privileges_enc"] = le_priv.transform(temp["privileges"])
    temp["product_enc"] = le_product.transform(temp["product"])
    
    X_input = temp[["cvss", "attack_vector_enc", "privileges_enc", "product_enc"]]
    
    # Predict risk
    temp["predicted_risk"] = model.predict(X_input)
    
    # Sort by CVSS (highest risk first)
    temp = temp.sort_values(by="cvss", ascending=False)
    
    return temp


# =========================================================
# 💻 9. MAIN SYSTEM LOOP (USER INTERACTION)
# =========================================================
while True:
    user_input = input("\nEnter tech stack (e.g. android 13.0) or 'exit': ")
    
    if user_input.lower() == "exit":
        break
    
    product, version = parse_input(user_input)
    
    # Find CVEs
    cves = find_cves(product, version)
    
    # Rank CVEs using ML
    ranked = rank_cves(cves)
    
    # Output
    if ranked.empty:
        print("✅ No vulnerabilities found")
    else:
        print("\n💀 Top Vulnerabilities:\n")
        
        for _, row in ranked.head(5).iterrows():
            print(f"""
CVE: {row['cve_id']}
Product: {row['product']}
Version Rule: {row['operator']} {row['version']}
CVSS: {row['cvss']}
Risk: {row['label']}
Predicted Risk: {row['predicted_risk']}
----------------------------------------
""")'''

' # =========================================================\n# 🔥 1. IMPORTS\n# =========================================================\nimport pandas as pd\nimport re\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.preprocessing import LabelEncoder\n\n\n# =========================================================\n# 📦 2. LOAD DATASET\n# =========================================================\ndf = pd.read_csv("dataset_fixed.csv")\n\n\n# =========================================================\n# 🔤 3. ENCODING (FOR ML MODEL)\n# =========================================================\nle_attack = LabelEncoder()\nle_priv = LabelEncoder()\nle_product = LabelEncoder()\n\ndf["attack_vector_enc"] = le_attack.fit_transform(df["attack_vector"])\ndf["privileges_enc"] = le_priv.fit_transform(df["privileges"])\ndf["product_enc"] = le_product.fit_transform(df["product"])\n\n\n# =========================================

In [4]:
# =========================================================
# 🔥 1. IMPORTS
# =========================================================
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder


# =========================================================
# 📦 2. LOAD DATASET
# =========================================================
df = pd.read_csv("dataset_fixed.csv")


# =========================================================
# 🔤 3. ENCODING (FOR ML MODEL)
# =========================================================
le_attack = LabelEncoder()
le_priv = LabelEncoder()
le_product = LabelEncoder()

df["attack_vector_enc"] = le_attack.fit_transform(df["attack_vector"])
df["privileges_enc"] = le_priv.fit_transform(df["privileges"])
df["product_enc"] = le_product.fit_transform(df["product"])


# =========================================================
# 🤖 4. TRAIN ML MODEL
# =========================================================
X = df[["cvss", "attack_vector_enc", "privileges_enc", "product_enc"]]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

print("✅ ML Model Trained")


# =========================================================
# 🧠 5. INPUT PARSER
# =========================================================
def parse_input(user_input):
    parts = user_input.lower().split()
    product = parts[0]
    version = parts[1] if len(parts) > 1 else None
    return product, version


# =========================================================
# 🔢 6. VERSION HANDLING
# =========================================================
def normalize(v, length=3):
    try:
        parts = [int(x) for x in v.split(".")]
        while len(parts) < length:
            parts.append(0)
        return tuple(parts)
    except:
        return (0,)

def get_generic_versions(v):
    parts = v.split(".")
    versions = [v]
    
    if len(parts) > 1:
        versions.append(".".join(parts[:-1]))
    
    versions.append(parts[0])
    
    return list(set(versions))


# =========================================================
# 🔍 7. CVE FINDER (WITH PRIORITY 🔥)
# =========================================================
def find_cves(product_input, version_input):
    results = df[df["product"].str.lower() == product_input.lower()]
    
    primary = []
    secondary = []
    
    user_versions = get_generic_versions(version_input)
    
    for _, row in results.iterrows():
        try:
            row_version = str(row["version"])
            
            for v in user_versions:
                user_v = normalize(v)
                row_v = normalize(row_version)
                
                matched = False
                
                if row["operator"] == "=" and user_v == row_v:
                    matched = True
                elif row["operator"] == "<=" and user_v <= row_v:
                    matched = True
                elif row["operator"] == "<" and user_v < row_v:
                    matched = True
                elif row["operator"] == ">=" and user_v >= row_v:
                    matched = True
                elif row["operator"] == ">" and user_v > row_v:
                    matched = True
                
                if matched:
                    row_copy = row.copy()
                    
                    # 🔥 PRIORITY LOGIC
                    if str(row_version).startswith(version_input):
                        row_copy["priority"] = 0  # exact / closest
                        primary.append(row_copy)
                    else:
                        row_copy["priority"] = 1  # secondary
                        secondary.append(row_copy)
        
        except:
            continue
    
    final_df = pd.DataFrame(primary + secondary).drop_duplicates()
    
    return final_df


# =========================================================
# 🧠 8. ML RANKING
# =========================================================
def rank_cves(cve_df):
    if cve_df.empty:
        return cve_df
    
    temp = cve_df.copy()
    
    temp["attack_vector_enc"] = le_attack.transform(temp["attack_vector"])
    temp["privileges_enc"] = le_priv.transform(temp["privileges"])
    temp["product_enc"] = le_product.transform(temp["product"])
    
    X_input = temp[["cvss", "attack_vector_enc", "privileges_enc", "product_enc"]]
    
    temp["predicted_risk"] = model.predict(X_input)
    
    # 🔥 PRIORITY FIRST, THEN CVSS
    temp = temp.sort_values(by=["priority", "cvss"], ascending=[True, False])
    
    return temp


# =========================================================
# 💻 9. MAIN LOOP
# =========================================================
while True:
    user_input = input("\nEnter tech stack (e.g. linux_kernel 6.6) or 'exit': ")
    
    if user_input.lower() == "exit":
        break
    
    product, version = parse_input(user_input)
    
    cves = find_cves(product, version)
    ranked = rank_cves(cves)
    
    if ranked.empty:
        print("✅ No vulnerabilities found")
    
    else:
        print("\n🎯 Exact / Closest Version Matches:\n")
        
        primary = ranked[ranked["priority"] == 0]
        secondary = ranked[ranked["priority"] == 1]
        
        # PRIMARY RESULTS
        for _, row in primary.head(3).iterrows():
            print(f"""
CVE: {row['cve_id']}
Product: {row['product']}
Version Rule: {row['operator']} {row['version']}
CVSS: {row['cvss']}
Risk: {row['label']}
----------------------------------------
""")
        
        # SECONDARY RESULTS
        if not secondary.empty:
            print("\n⚠️ Other Related Vulnerabilities:\n")
            
            for _, row in secondary.head(3).iterrows():
                print(f"""
CVE: {row['cve_id']}
Product: {row['product']}
Version Rule: {row['operator']} {row['version']}
CVSS: {row['cvss']}
Risk: {row['label']}
----------------------------------------
""")

✅ ML Model Trained



Enter tech stack (e.g. linux_kernel 6.6) or 'exit':  mongodb 5.0



🎯 Exact / Closest Version Matches:


CVE: CVE-2024-7553
Product: mongodb
Version Rule: < 5.0.27
CVSS: 7.3
Risk: MEDIUM
----------------------------------------


CVE: CVE-2024-8207
Product: mongodb
Version Rule: < 5.0.14
CVSS: 6.4
Risk: MEDIUM
----------------------------------------


CVE: CVE-2024-6375
Product: mongodb
Version Rule: < 5.0.22
CVSS: 5.4
Risk: MEDIUM
----------------------------------------


⚠️ Other Related Vulnerabilities:


CVE: CVE-2024-7553
Product: mongodb
Version Rule: < 6.0.16
CVSS: 7.3
Risk: MEDIUM
----------------------------------------


CVE: CVE-2024-7553
Product: mongodb
Version Rule: < 7.0.12
CVSS: 7.3
Risk: MEDIUM
----------------------------------------


CVE: CVE-2024-7553
Product: mongodb
Version Rule: < 7.3.3
CVSS: 7.3
Risk: MEDIUM
----------------------------------------




Enter tech stack (e.g. linux_kernel 6.6) or 'exit':  exit
